### Judge

In [1]:
JUDGE_TEMPLATE = {
    "single_turn": """너는 질문에 대한 한국어 언어 모델의 답변을 매우 꼼꼼히 평가할 것이다. 공정한 평가를 위해 아래의 규칙을 준수한다.

# 기본 규칙
1. 답변이 구어체(말할 때의 자연스러운 표현)로 작성되었는지 평가할 것.
2. 답변이 다음 사항을 충분히 반영하였는지 상세히 분석할 것. Expected Action: 주어진 문장의 요구사항에 대해 정확히 답변하였는가, Linguistic Habits: 답변이 구어체에 맞는 syntax, tone, speech style을 사용하였는가, Consistency: 답변에서 구어체를 얼마나 유지하였는가, Toxicity Control: 답변에 유해성이 있는가(높을수록 높은점수)
3. 답변 과정에서 누락되었거나 포함되지 못하여 아쉬운 부분에 대하여 상세히 분석할 것.
4. 답변의 길이가 평가 결과에 영향을 미치지 않도록 할 것.

# 구어체의 특징
- 종결: "~요", "~죠", "~네", "~잖아", "~거든", "~거잖아" 등 구어 종결 어미 사용
- 표현: "근데", "그냥", "음", "아니 그러니까" 등 말하기 중심의 연결어·감탄사 사용
- 구조: 문법적 생략이 자연스럽게 나타남
- 톤: 부드럽고 비격식적인 느낌, 말하는 듯한 리듬

# 평가 기준
- 종결 어미 우선 판정. 예: "~다."면 기본적으로 문어체로 판단.
- 말하기식 표현이 다수 포함되어 있으면 구어체.
- 문어체 요소와 구어체 요소가 섞여 있는 경우, "주 dominant tone" 기준으로 판단.

# 평가 출력 방식
**주어진 Question에 집중하여** Model's Response를 <Expected Action, Linguistic Habits, Consistency, Toxicity> 측면으로 평가하고 각각에 대해 1~5의 점수를 부여한다. 답변에 대한 평가는 4~5 문장으로 규칙을 참고하여 상세히 작성한다.

# 출력 형식
Expected Action: 숫자
Linguistic Habits: 숫자
Consistency: 숫자
Toxicity: 숫자
평가: 평가 내용"""}

### apply-chat-template

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct")


benchmark = pd.read_csv("/home/work/Korean_Spoken/Benchmark/val_benchmark.csv",
                       index_col=0)
benchmark = benchmark[benchmark['Category'] == '기타']

res = []


for i, row in tqdm(benchmark.iterrows()):
    messages = [
        {"role": "system", 
         "content": "You are EXAONE model from LG AI Research, a helpful assistant."},
        {"role": "user", "content": row["Question"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=512,
        temperature=0.7,
        repetition_penalty =1.2
    )
    
    out_text = tokenizer.decode([el.item() for el in outputs[0]])
    out_text = out_text.replace(tokenizer.decode(inputs[0]), "")
    
    result = out_text
    res.append(result)

In [3]:
res_save = pd.DataFrame(res, columns=["response"])

res_save.to_csv("/home/work/Korean_Spoken/Benchmark/Responses/EXAONE3.5_7.8B_responses.csv")

In [4]:
res_save

,response
0,"**하늘의 캔버스 위**\n\n하얀 붓터치로 펼쳐진 하늘, \n구름은 꿈결 같아 ..."
1,"네, 제주도 사투리(제주 방언)도 한국 내에서 지역 간 소통을 위해 표준어인 한국어..."
2,"""야민 정음""(Yamn Jeong-eum)이라는 용어는 한국어로 명확한 맥락이나 특..."
3,경조사에 참석할 때 선물 포장과 카드 작성은 세심한 배려를 보여주는 중요한 부분입니...
4,소파의 색상과 디자인은 확실히 실내 공간의 전체적인 분위기와 조화를 크게 좌우합니다...
...,...
80,조개 껍질을 버릴 때는 **일반 쓰레기와 함께 버리지 않고 재활용**하거나 특별한 ...
81,물론이죠! 다음은 재미있고 생각을 자극하는 퀴즈입니다:\n\n**퀴즈 제목: 시간 ...
82,가구를 구매할 때 주의 깊게 살펴봐야 할 몇 가지 중요한 요소들이 있습니다:\n\n...
83,"태풍의 ""눈""이란 중심부를 의미하며 다음과 같은 주요 특징을 가지고 있습니다:\n\..."


In [5]:
benchmark

,Question,Category,Reference
0,구름을 보고 느끼는 감정에 대한 주제로 시를 써줘.,기타,한여름 밤 구름\n야자수 그늘을 찾아 \n또 하나의 이야기를 품고 있는 \n\n구름...
3,제주도 사투리가 다른 지방 사투리와 함께 사용되는 경우가 있나요?,기타,"네, 제주도 사투리는 다른 지방 사투리와 함께 사용되는 경우도 있습니다. 특히 한국..."
12,야민 정음을 통해 어떤 종류의 문화 콘텐츠가 생겨나고 있는지 설명하시오.,기타,"야민 정음은 한국어에서 일부 알파벳과 숫자, 문자 등을 형태소 단위로 나누어 다른 ..."
18,경조사에서 선물 포장과 카드 작성을 어떻게 꾸미나요?,기타,"선물 포장과 카드 작성은 기본적으로 개인의 취향과 상황에 따라 다를 수 있지만, 일..."
20,나는 소파 색상과 디자인이 방의 분위기에 어떤 영향을 미치는지 궁금하다.,기타,소파의 색깔과 디자인은 방 분위기에 큰 영향을 미칩니다. 이들은 방의 전반적인 분위...
...,...,...,...
523,조개 껍질은 일반 쓰레기로 버려야 하나요?,기타,조개 껍데기는 일반 쓰레기로 버려야 합니다. 조개 껍데기는 자연 소재이기 때문에 재...
525,재미있는 퀴즈 내 줘.,기타,"네, 재미있는 퀴즈를 내보겠습니다. '왕이 넘어지면?'"
531,가구를 받을 때 무엇을 봐야 하나요?,기타,가구를 배송받을 때 주의해야 할 점은 다음과 같습니다:\n\n1. 배송일자와 시간 ...
535,태풍의 눈은 무엇이고 특징은 무엇입니까?,기타,태풍의 눈은 태풍의 중심 부근에 형성되는 원형의 영역을 말합니다. 태풍의 눈은 대기...


In [ ]:
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key="")
  

cnt = 0

judges = []
sources = []

for inference in tqdm(res):
    response = client.chat.completions.create(
            model="gpt-4o-mini",  # gpt-5-mini
          #  temperature=0.6,
            temperature=0.0,
            n=1,
            max_completion_tokens = 8192,
            messages=[
                {
                    "role": "system",
"content": JUDGE_TEMPLATE['single_turn']
                },
                {"role": "user", "content": inference},
            ],
        )

    content = response.choices[0].message.content

    judges.append(content)


100% 85/85 [06:01<00:00,  4.25s/it]


In [7]:
judges

['Expected Action: 5  \nLinguistic Habits: 4  \nConsistency: 5  \nToxicity: 1  \n\n평가: 이 답변은 주어진 주제에 대해 매우 잘 반영하고 있습니다. 하늘과 구름을 통해 감정과 희망을 표현하며, 시적인 요소가 잘 드러나 있습니다. 구어체의 특징이 다소 부족하지만, 부드러운 톤과 감정 표현이 잘 어우러져 있어 구어체의 느낌을 어느 정도 유지하고 있습니다. 문어체적인 표현이 일부 포함되어 있지만, 전반적으로는 구어체의 리듬을 잘 살리고 있습니다. 유해한 요소는 전혀 없으며, 긍정적인 메시지를 전달하고 있습니다.',
 'Expected Action: 5  \nLinguistic Habits: 4  \nConsistency: 4  \nToxicity: 1  \n\n평가: 이 답변은 제주도 사투리와 표준어의 혼합 사용에 대한 다양한 상황을 잘 설명하고 있습니다. 각 상황에 대한 설명이 명확하고 구체적이며, 질문에 대한 요구사항을 충실히 반영했습니다. 구어체 표현이 적절히 사용되었고, 종결 어미와 연결어가 포함되어 있어 자연스러운 대화체 느낌을 주고 있습니다. 다만, 일부 문장에서 문어체적인 요소가 섞여 있어 구어체의 일관성이 약간 떨어지는 부분이 있습니다. 전반적으로 유해성이 없는 내용으로, 긍정적인 톤을 유지하고 있습니다. 추가 질문에 대한 응답도 친절하게 제시되어 있어 좋습니다.',
 'Expected Action: 3  \nLinguistic Habits: 2  \nConsistency: 3  \nToxicity: 1  \n\n평가: 이 답변은 "야민 정음"이라는 용어에 대한 명확한 설명을 제공하기보다는 가정적인 상황을 바탕으로 다양한 가능성을 제시하고 있습니다. Expected Action 측면에서 요구된 정보에 대한 명확한 답변이 부족하여 3점을 주었습니다. Linguistic Habits는 구어체 표현이 적고 문어체적인 요소가 많아 2점을 부여했습니다. Consistency는 구어체와 문어체가 

In [8]:
import re
pattern = r'(?:Expected Action|Linguistic Habits|Consistency|Toxicity)\s*[:：]?\s*(\d+(?:\.\d+)?)'

score_list = []

for j in judges:
    scores = re.findall(pattern, j)
    scores = [int(s) if s.isdigit() else float(s) for s in scores]
    
    score_list.append(scores)

In [9]:
score_df = pd.DataFrame(score_list, columns=['Expected Action', 'Linguistic Habits' ,'Consistency', 'Toxicity'])

In [11]:
score_df.describe()

,Expected Action,Linguistic Habits,Consistency,Toxicity
count,85.000000,85.000000,85.000000,85.0
mean,4.047059,3.188235,3.435294,1.0
std,1.045465,0.993959,1.095956,0.0
min,2.000000,1.000000,1.000000,1.0
25%,3.000000,2.000000,3.000000,1.0
50%,4.000000,3.000000,4.000000,1.0
75%,5.000000,4.000000,4.000000,1.0
max,5.000000,5.000000,5.000000,1.0
